# Anti-Deepfake-Box — Full GPU Pipeline (Colab)

**Runtime**: Runtime → Change runtime type → **T4 GPU**

## What this notebook does
1. Install all dependencies + ffmpeg
2. Clone required repos
3. Download checkpoints (SyncNet from HuggingFace; XceptionNet from Drive or HF)
4. Mount Google Drive for FF++ dataset
5. Calibrate SNR threshold on FF++ val
6. Run full evaluation on FF++ test set
7. (Optional) Train meta-classifier fusion
8. Push trained weights to GitHub

## Before you start
Add these to **Colab Secrets** (key icon on left sidebar):
- `GITHUB_TOKEN` — GitHub Personal Access Token (repo scope)
- `HF_TOKEN` — HuggingFace token (optional, for gated models)

## 1. GPU Check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU')

## 2. Install Dependencies

In [ ]:
# System packages
!apt-get install -y ffmpeg > /dev/null 2>&1
!ffmpeg -version 2>&1 | head -1

# Python packages (torch already in Colab)
!pip install -q \
    insightface \
    onnxruntime-gpu \
    opencv-python-headless \
    scipy \
    scikit-learn \
    openai-whisper \
    librosa \
    decord \
    tqdm \
    Pillow \
    PyYAML \
    huggingface_hub

print('Done.')

## 3. Clone Repos

In [ ]:
import os

REPOS = {
    'anti-deepfake-box': 'https://github.com/nia1003/anti-deepfake-box.git',
    'rPPG-Toolbox':      'https://github.com/nia1003/rppg-toolbox.git',
    'FaceForensics':     'https://github.com/nia1003/faceforensics.git',
    'LatentSync':        'https://github.com/nia1003/latentsync.git',
}

for name, url in REPOS.items():
    if not os.path.exists(f'/content/{name}'):
        !git clone --depth 1 {url} /content/{name}
    else:
        print(f'{name}: already cloned')

# Set PYTHONPATH
import sys
for p in [
    '/content/anti-deepfake-box',
    '/content/rPPG-Toolbox',
    '/content/FaceForensics/classification',
    '/content/LatentSync',
]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir('/content/anti-deepfake-box')
print('Working directory:', os.getcwd())

## 4. Download Checkpoints

In [ ]:
import os, shutil
from pathlib import Path

os.makedirs('checkpoints', exist_ok=True)

# ── 4a. PhysNet (already in rPPG-Toolbox repo) ──────────────────────────
physnet_src = '/content/rPPG-Toolbox/final_model_release/MA-UBFC_physnet.pth'
physnet_dst = 'checkpoints/physnet_ubfc.pth'
if not Path(physnet_dst).exists():
    shutil.copy(physnet_src, physnet_dst)
print(f'PhysNet : {physnet_dst} ({Path(physnet_dst).stat().st_size/1e6:.1f} MB)')

# ── 4b. SyncNet (ByteDance/LatentSync-1.6 on HuggingFace) ───────────────
syncnet_dst = 'checkpoints/stable_syncnet.pt'
if not Path(syncnet_dst).exists():
    from huggingface_hub import hf_hub_download
    import os
    hf_token = None
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass
    path = hf_hub_download(
        repo_id='ByteDance/LatentSync-1.6',
        filename='stable_syncnet.pt',
        local_dir='checkpoints',
        token=hf_token,
    )
    print(f'SyncNet : downloaded to {path}')
else:
    print(f'SyncNet : {syncnet_dst} ({Path(syncnet_dst).stat().st_size/1e6:.1f} MB)')

# ── 4c. XceptionNet ─────────────────────────────────────────────────────
# Option A: load from Google Drive (if you have the FF++ c23 checkpoint)
# Option B: falls back to EfficientNet-B0 (automatic in visual_detector.py)
xception_drive = '/content/drive/MyDrive/checkpoints/xception_ff_c23.pth'
xception_dst   = 'checkpoints/xception_ff_c23.pth'
if not Path(xception_dst).exists():
    if Path(xception_drive).exists():
        shutil.copy(xception_drive, xception_dst)
        print(f'XceptionNet: loaded from Drive')
    else:
        print('XceptionNet: not found — will use EfficientNet-B0 fallback')
        print('  To use XceptionNet, upload checkpoint to:')
        print('  Google Drive → checkpoints/xception_ff_c23.pth')
else:
    print(f'XceptionNet: {xception_dst} ({Path(xception_dst).stat().st_size/1e6:.1f} MB)')

## 5. Mount Google Drive (FF++ Dataset)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set your FF++ root path here
FF_ROOT = '/content/drive/MyDrive/datasets/FaceForensics'

import os
if os.path.exists(FF_ROOT):
    print(f'FF++ found at: {FF_ROOT}')
    !ls {FF_ROOT}
else:
    print(f'WARNING: FF++ not found at {FF_ROOT}')
    print('Set FF_ROOT to the correct path before continuing.')

## 6. Update Config for GPU

In [ ]:
import yaml
from pathlib import Path

cfg_path = Path('configs/default.yaml')
cfg = yaml.safe_load(cfg_path.read_text())

cfg['device'] = 'cuda'
cfg['detectors']['visual']['device'] = 'cuda'
cfg['detectors']['rppg']['device']   = 'cuda'
cfg['detectors']['sync']['device']   = 'cuda'

cfg['detectors']['visual']['pretrained']  = 'checkpoints/xception_ff_c23.pth'
cfg['detectors']['rppg']['pretrained']    = 'checkpoints/physnet_ubfc.pth'
cfg['detectors']['sync']['syncnet_path']  = 'checkpoints/stable_syncnet.pt'

cfg_path.write_text(yaml.dump(cfg, default_flow_style=False))
print('Config updated:')
print(yaml.dump(cfg, default_flow_style=False))

## 7. Quick Smoke Test
Run on a LatentSync demo video to verify the pipeline works end-to-end before processing FF++.

In [ ]:
!python scripts/inference.py \
    --video /content/LatentSync/assets/demo1_video.mp4 \
    --config configs/default.yaml \
    --async_mode

## 8. Calibrate SNR Threshold (FF++ val)
Finds the optimal `snr_threshold` using Youden's J statistic and writes it back to config.

In [ ]:
FF_ROOT = '/content/drive/MyDrive/datasets/FaceForensics'  # change if needed

!python scripts/calibrate_snr.py \
    --data_root {FF_ROOT} \
    --config configs/default.yaml \
    --split val \
    --compression c23 \
    --update_config

# Show updated threshold
import yaml
cfg = yaml.safe_load(open('configs/default.yaml'))
print('Calibrated SNR threshold:', cfg['detectors']['rppg']['snr_threshold'])

## 9. Evaluate on FF++ Test Set
Reports AUC, ACC, EER, AP per detector and for the ensemble.

In [ ]:
import os
os.makedirs('results', exist_ok=True)
FF_ROOT = '/content/drive/MyDrive/datasets/FaceForensics'

!python scripts/evaluate.py \
    --dataset ff++ \
    --data_root {FF_ROOT} \
    --compression c23 \
    --config configs/default.yaml \
    --split test \
    --output results/ff_c23_test.json

import json
results = json.load(open('results/ff_c23_test.json'))
print('\n=== Evaluation Results (FF++ c23 test) ===')
for k, v in results.items():
    print(f'  {k}: {v}')

## 10. (Optional) Train Meta-Classifier
Only run this if AUC from Step 9 is > 0.95 on FF++ test.

### Stage 1: Grid search for best fusion weights

In [ ]:
# Check gate condition
import json
results = json.load(open('results/ff_c23_test.json'))
auc = results.get('ensemble', {}).get('auc', 0)
print(f'Ensemble AUC: {auc:.4f}')

if auc >= 0.95:
    print('Gate passed (AUC >= 0.95). Proceeding to Stage 1 weight search...')
    FF_ROOT = '/content/drive/MyDrive/datasets/FaceForensics'
    !python scripts/train_fusion.py \
        --config configs/training.yaml \
        --stage weighted \
        --data_root {FF_ROOT} \
        --output results/stage1_weights.json
else:
    print(f'Gate NOT passed (AUC={auc:.4f} < 0.95).')
    print('Investigate which detector is underperforming before training meta-classifier.')

### Stage 2: Train MLP meta-classifier

In [ ]:
import os
os.makedirs('results', exist_ok=True)
FF_ROOT = '/content/drive/MyDrive/datasets/FaceForensics'

!python scripts/train_fusion.py \
    --config configs/training.yaml \
    --stage meta \
    --data_root {FF_ROOT} \
    --scores_cache results/scores_cache.npz \
    --epochs 30 \
    --output checkpoints/meta_classifier_best.pth

from pathlib import Path
p = Path('checkpoints/meta_classifier_best.pth')
if p.exists():
    print(f'Meta-classifier saved: {p} ({p.stat().st_size/1e6:.2f} MB)')

## 11. Push Results to GitHub
Pushes:
- `results/ff_c23_test.json` — evaluation metrics
- `checkpoints/meta_classifier_best.pth` — trained MLP (if Stage 2 ran)
- Updated `configs/default.yaml` — calibrated SNR threshold

In [ ]:
from google.colab import userdata
import subprocess, os

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USER  = 'nia1003'
REPO_NAME    = 'anti-deepfake-box'
BRANCH       = 'main'

# Configure git
!git config user.email 'colab@anti-deepfake-box'
!git config user.name  'Colab Pipeline'

# Set remote with token
remote_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
!git remote set-url origin {remote_url}

# Pull latest
!git pull origin {BRANCH} --rebase

# Stage result files (force-add checkpoints despite .gitignore)
!git add -f configs/default.yaml
!git add -f results/ff_c23_test.json 2>/dev/null || true

# Only add meta-classifier if it exists and is small enough (<100MB GitHub limit)
from pathlib import Path
meta_path = Path('checkpoints/meta_classifier_best.pth')
if meta_path.exists() and meta_path.stat().st_size < 95_000_000:
    !git add -f checkpoints/meta_classifier_best.pth
    print(f'Staging meta_classifier_best.pth ({meta_path.stat().st_size/1e6:.1f} MB)')

# Commit
from datetime import datetime
date_str = datetime.now().strftime('%Y-%m-%d')
!git commit -m "results: FF++ c23 evaluation + calibrated SNR ({date_str}) [Colab GPU]"

# Push
!git push origin {BRANCH}
print('\nPushed to GitHub.')

## 12. Summary
Print a final summary of everything that ran.

In [ ]:
import json, os
from pathlib import Path

print('=' * 60)
print('ANTI-DEEPFAKE-BOX — COLAB RUN SUMMARY')
print('=' * 60)

# Checkpoints
print('\nCheckpoints:')
for ck in ['checkpoints/physnet_ubfc.pth',
           'checkpoints/stable_syncnet.pt',
           'checkpoints/xception_ff_c23.pth',
           'checkpoints/meta_classifier_best.pth']:
    p = Path(ck)
    status = f'{p.stat().st_size/1e6:.1f} MB' if p.exists() else 'NOT FOUND'
    print(f'  {ck:<45} {status}')

# Results
if Path('results/ff_c23_test.json').exists():
    results = json.load(open('results/ff_c23_test.json'))
    print('\nEvaluation (FF++ c23 test):')
    for detector, metrics in results.items():
        if isinstance(metrics, dict):
            auc = metrics.get('auc', 'N/A')
            acc = metrics.get('acc', 'N/A')
            print(f'  {detector:<20} AUC={auc:.4f}  ACC={acc:.4f}')
else:
    print('\nNo evaluation results yet (Step 9 not run).')

# Config
import yaml
cfg = yaml.safe_load(open('configs/default.yaml'))
print(f"\nCalibrated SNR threshold: {cfg['detectors']['rppg']['snr_threshold']}")
print(f"Fusion mode: {cfg['fusion']['mode']}")
print('=' * 60)